# DLR Carrier Board — Power & Signal Integrity Analysis

Single-PCB CM4 carrier for an outdoor solar-powered RTU. Soldered Quectel BG770A (LTE Cat-M1, NA bands), FLIR Lepton 3.5 thermal imager (SPI), DHT22 + SI1145 + ADS1115/YL-83 sensors, MPPT charger from 20W solar panel, LiFePO4 4S battery + BMS, buck for 5V SoM rail, AP2112K LDO for 3.3V analog.

This notebook derives the buck input current envelope, daily energy budget, I2C pull-up sizing, SPI critical trace length, and USB 2.0 differential pair geometry.

## Assumptions
- LiFePO4 4S battery: V_min=10.0V (2.5V/cell cutoff), V_nom=12.8V, V_max=14.6V (3.65V/cell charge top)
- 4 Ah battery cell — 50 Wh nominal, 90% usable depth-of-discharge per LiFePO4 chemistry tolerance
- 20W solar panel rated at STC; winter NE US worst case = 2.5 sun-hours/day, annual avg 4.0
- Generic synchronous buck regulator, η=0.90±0.03 across the 10–14.6V Vin range
- AP2112K LDO @ 3.3V — datasheet dropout 250 mV at 600 mA (we run ~14 mA peak)
- CM4 always-on idle baseline: 120 mA at 5V (Pi OS Lite, headless, no WiFi/BT)
- Duty cycle: sensor wake 5s every 60s, cellular TX 5s every 15 min; CM4 in idle the rest of the time
- I2C bus cap: 4 device slots × 10pF + ~30pF traces + 5pF connector = 75pF estimate
- FR4 substrate εr=4.3; v_prop = c/√εr ≈ 1.45×10⁸ m/s
- 4-layer 1.6mm board, F.Cu to In1.GND prepreg height = 0.20 mm (controlled-impedance fab order)
- Parasitic inductance and trace resistance neglected for power rails (short runs, bulk caps present)

### Wind sensor (Section 6) — additional assumptions
- IEEE 738-2012 (R2023) §4.4.3 — Hilpert cross-flow Nusselt + Eq 4a wind-angle correction is the authoritative model. Use the implementation in `dlr-operating-envelope/src/ieee738.py` verbatim — no re-derivation.
- Worst-case rating operating point per Annex A worked example and `ieee738_test.py:114`: ACSR Drake 26/7, V_w = 0.61 m/s, φ = 90°, T_c = 100°C, T_a = 40°C, q_s = 0.
- Acceptable line-rating uncertainty from wind sensing alone: **±5 % of I_max** — derived from utility DLR practice where total ampacity uncertainty budget is ~10 % and the wind term is the dominant single contributor.
- Sensor mount: ~1–2 m above the conductor on the same cross-arm, in clean airflow. Cable run from sensor to PCB: 2 m max.
- 60 Hz B-field near a 1000 A conductor at 1 m radius: B ≈ μ₀·I/(2π·r) ≈ 0.2 mT — strong enough that unshielded single-ended I2C is not viable; differential serial (RS-485) or 4-20 mA loop required.
- Icing-fallback policy: if a non-heated sensor is selected, an iced-transducer condition reports `None`/error to `dlr-operating-envelope`, which MUST fall back to the conductor's static thermal rating (V_w = 0 equivalent) until the sensor recovers. This is the conservative direction; loss of headroom but no over-temp risk.
- Heated sensor (if selected) heater duty cycle is on-demand only when transducer self-test detects ice, NOT continuous. Modeled as worst-case 2 hours/day in winter for power budgeting.

In [1]:
"""Universal constants for DLR carrier theory derivation."""

import math
from typing import Final

import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

# === Battery: LiFePO4 4S ===
# Reason: 3.2V/cell nominal x 4 = 12.8V; 2.5V cutoff x 4 = 10.0V; 3.65V charge x 4 = 14.6V
V_BAT_NOM: Final = 12.8 * ureg.V
V_BAT_MIN: Final = 10.0 * ureg.V
V_BAT_MAX: Final = 14.6 * ureg.V
BAT_CAPACITY: Final = 4.0 * ureg.A * ureg.hour
BAT_USABLE_FRACTION: Final = 0.90

# === Buck (5V rail) and LDO (3V3 rail) ===
V_RAIL_5V: Final = 5.0 * ureg.V
V_RAIL_3V3: Final = 3.3 * ureg.V
BUCK_EFFICIENCY: Final = (
    ufloat(0.90, 0.03) * ureg.dimensionless
)  # generic synchronous buck
LDO_DROPOUT_AT_600MA: Final = 250 * ureg.mV  # AP2112K datasheet

# === Solar / power budget ===
PANEL_RATING: Final = 20 * ureg.W
SUN_HOURS_WORST: Final = 2.5 * ureg.hour  # December NE US
SUN_HOURS_AVG: Final = 4.0 * ureg.hour  # Annual avg NE US
MPPT_EFFICIENCY: Final = 0.90
I_5V_PEAK: Final = 3920 * ureg.mA
I_5V_TYPICAL: Final = 1660 * ureg.mA

# === I2C bus (UM10204 Rev 7.0 Table 10, fast mode 400 kHz) ===
V_OL_MAX: Final = 0.4 * ureg.V
I_OL_FAST: Final = 3.0 * ureg.mA
T_RISE_FAST_MAX: Final = 300 * ureg.ns
C_BUS_ESTIMATED: Final = 75 * ureg.pF

# === FLIR Lepton 3.5 SPI ===
F_SPI_MAX: Final = 20 * ureg.MHz
BW_HARMONIC_FACTOR: Final = 5  # 5x f_clk for clean edges

# === FR4 substrate + USB 2.0 ===
EPSILON_R_FR4: Final = 4.3
V_PROP_FR4: Final = 3e8 / math.sqrt(EPSILON_R_FR4) * ureg.m / ureg.s
Z_DIFF_USB_TARGET: Final = 90 * ureg.ohm
USB_SKEW_MAX: Final = 100 * ureg.ps
H_TO_GND: Final = 0.20 * ureg.mm  # F.Cu to In1.GND on 4L 1.6mm board
W_TRACE_USB: Final = 0.20 * ureg.mm  # 8 mil
G_GAP_USB: Final = 0.15 * ureg.mm  # 6 mil

print("Constants loaded.")

Constants loaded.


## Section 1: Power Architecture

Buck input current via power conservation. Worst case is at minimum battery voltage (post-cutoff approach):

$$ I_{buck,in} = \frac{V_{out} \cdot I_{out}}{V_{in} \cdot \eta} $$

LDO V_in headroom must clear (V_out + V_dropout) at peak load:

$$ \Delta V_{headroom} = V_{rail,5V} - V_{rail,3V3} - V_{dropout} $$

In [2]:
# Buck input current envelope
i_in_peak_worst = (V_RAIL_5V * I_5V_PEAK) / (V_BAT_MIN * BUCK_EFFICIENCY)
i_in_typ_nom = (V_RAIL_5V * I_5V_TYPICAL) / (V_BAT_NOM * BUCK_EFFICIENCY)
i_bat_1c = BAT_CAPACITY / (1 * ureg.hour)

print(f"I_buck_in (peak @ V_BAT_MIN): {i_in_peak_worst.to(ureg.mA):.0f}")
print(f"I_buck_in (typ @ V_BAT_NOM): {i_in_typ_nom.to(ureg.mA):.0f}")
print(f"Battery 1C discharge limit: {i_bat_1c.to(ureg.A):.1f}")

# LDO headroom check
v_ldo_headroom = V_RAIL_5V - V_RAIL_3V3 - LDO_DROPOUT_AT_600MA
print(f"\nLDO V_in headroom: {v_ldo_headroom.to(ureg.V):.2f}")

I_buck_in (peak @ V_BAT_MIN): 2178+/-73 milliampere
I_buck_in (typ @ V_BAT_NOM): 720+/-24 milliampere
Battery 1C discharge limit: 4.0 ampere

LDO V_in headroom: 1.45 volt


## Section 2: Daily Energy Budget

Realistic operating profile — CM4 always-on at idle, brief active wakes for sampling, cellular bursts every 15 min. Average current is dominated by CM4 idle.

$$ I_{avg} = I_{idle} + \sum_i (I_i - I_{idle}) \cdot D_i $$

where $D_i = t_{on,i} / t_{period,i}$ is the duty cycle of each consumer.

In [3]:
# Duty cycle constants
SAMPLE_INTERVAL: Final = 1 * ureg.minute
SAMPLE_DURATION: Final = 5 * ureg.s
TX_INTERVAL: Final = 15 * ureg.minute
TX_DURATION: Final = 5 * ureg.s

# Per-component currents at 5V
I_CM4_IDLE: Final = 120 * ureg.mA
I_CM4_ACTIVE: Final = 1400 * ureg.mA
I_LEPTON_ON: Final = 150 * ureg.mA
I_BG770_PSM: Final = 1 * ureg.mA
I_BG770_TX: Final = 250 * ureg.mA
I_SENSORS_ACTIVE: Final = 9 * ureg.mA

sample_duty = (SAMPLE_DURATION / SAMPLE_INTERVAL).to(ureg.dimensionless)
tx_duty = (TX_DURATION / TX_INTERVAL).to(ureg.dimensionless)

i_avg = (
    I_CM4_IDLE
    + (I_CM4_ACTIVE - I_CM4_IDLE) * sample_duty
    + I_LEPTON_ON * sample_duty
    + I_BG770_PSM
    + (I_BG770_TX - I_BG770_PSM) * tx_duty
    + I_SENSORS_ACTIVE * sample_duty
)
p_avg = i_avg * V_RAIL_5V
e_daily = p_avg * (24 * ureg.hour)

print(f"Sample duty: {sample_duty.magnitude:.4f}")
print(f"TX duty: {tx_duty.magnitude:.4f}")
print(f"Average current: {i_avg.to(ureg.mA):.0f}")
print(f"Average power: {p_avg.to(ureg.W):.2f}")
print(f"Daily energy: {e_daily.to(ureg.W * ureg.hour):.1f}")

# Solar production
e_pv_winter = PANEL_RATING * SUN_HOURS_WORST * MPPT_EFFICIENCY
e_pv_avg = PANEL_RATING * SUN_HOURS_AVG * MPPT_EFFICIENCY
winter_margin = (e_pv_winter / e_daily).to(ureg.dimensionless)

print(f"\nPV winter (worst): {e_pv_winter.to(ureg.W * ureg.hour):.1f}")
print(f"PV annual avg: {e_pv_avg.to(ureg.W * ureg.hour):.1f}")
print(f"Winter margin: {winter_margin.magnitude:.2f}x")

# Battery autonomy at zero PV
# Reason: V_BAT * BAT_CAPACITY (A*hr) gives Wh directly; no extra divide
e_bat_usable = V_BAT_NOM * BAT_CAPACITY * BAT_USABLE_FRACTION
days_autonomy = (e_bat_usable / e_daily).to(ureg.dimensionless)
print(f"\nBattery usable energy: {e_bat_usable.to(ureg.W * ureg.hour):.1f}")
print(f"Autonomy at zero PV: {days_autonomy.magnitude:.2f} days")

Sample duty: 0.0833
TX duty: 0.0056
Average current: 242 milliampere
Average power: 1.21 watt
Daily energy: 29.1 hour * watt

PV winter (worst): 45.0 hour * watt
PV annual avg: 72.0 hour * watt
Winter margin: 1.55x

Battery usable energy: 46.1 hour * watt
Autonomy at zero PV: 1.58 days


## Section 3: I2C Pull-Up Sizing (400 kHz, 3.3V)

Per UM10204 Rev 7.0 Table 10, fast mode requires R within bounds set by sink current and rise time:

$$ R_{min} = \frac{V_{cc} - V_{OL,max}}{I_{OL}} \qquad R_{max} = \frac{t_{rise,max}}{0.8473 \cdot C_{bus}} $$

The 0.8473 factor is the natural log ratio for RC charging from 30% to 70% of V_cc.

In [4]:
r_pullup_min = (V_RAIL_3V3 - V_OL_MAX) / I_OL_FAST
r_pullup_max = T_RISE_FAST_MAX / (0.8473 * C_BUS_ESTIMATED)

R_PULLUP_SELECTED: Final = 2.2 * ureg.kohm  # E24 standard, in range
t_rise_actual = 0.8473 * R_PULLUP_SELECTED * C_BUS_ESTIMATED

print(f"R_min: {r_pullup_min.to(ureg.ohm):.0f}")
print(f"R_max: {r_pullup_max.to(ureg.ohm):.0f}")
print(f"R_selected (E24): {R_PULLUP_SELECTED}")
print(f"In range: {r_pullup_min < R_PULLUP_SELECTED < r_pullup_max}")
print(
    f"Rise time at selected R: {t_rise_actual.to(ureg.ns):.0f} (max {T_RISE_FAST_MAX})"
)

R_min: 967 ohm
R_max: 4721 ohm
R_selected (E24): 2.2 kiloohm
In range: True
Rise time at selected R: 140 nanosecond (max 300 nanosecond)


## Section 4: SPI Signal Integrity (FLIR Lepton, 20 MHz)

Lepton 3.5 SPI runs at 20 MHz. Edge rise time required for clean edges (5x harmonic content):

$$ t_r \le \frac{0.35}{5 \cdot f_{clk}} $$

Critical trace length above which transmission-line behavior matters:

$$ L_{crit} = \frac{t_r \cdot v_{prop}}{2} $$

Design rule: keep traces below $L_{crit}/5$ to be safely lumped.

In [5]:
bw_required = BW_HARMONIC_FACTOR * F_SPI_MAX
t_rise_spi = 0.35 / bw_required
l_crit_spi = (t_rise_spi * V_PROP_FR4 / 2).to(ureg.mm)
l_design_rule = (l_crit_spi / 5).to(ureg.mm)

print(f"Required BW (5x f_clk): {bw_required.to(ureg.MHz):.0f}")
print(f"Edge rise time spec: {t_rise_spi.to(ureg.ns):.2f}")
print(f"Critical TL length: {l_crit_spi:.0f}")
print(f"Design rule (lumped trace budget): < {l_design_rule:.0f}")

Required BW (5x f_clk): 100 megahertz
Edge rise time spec: 3.50 nanosecond
Critical TL length: 253 millimeter
Design rule (lumped trace budget): < 51 millimeter


## Section 5: USB 2.0 Differential Routing (BG770A)

USB 2.0 High-Speed requires Z_diff = 90Ω ± 10% on the D+/D- pair, with intra-pair skew under 100 ps.

Single-ended microstrip impedance (Wadell):

$$ Z_0 = \frac{60}{\sqrt{\varepsilon_r}} \ln\left(\frac{8h}{w} + \frac{w}{4h}\right) $$

Edge-coupled differential approximation:

$$ Z_{diff} \approx 2 \cdot Z_0 \cdot \left(1 - 0.48 \cdot e^{-0.96 \cdot s/h}\right) $$

where $s$ is the gap between traces and $h$ is the height to the GND reference plane.

In [6]:
# Strip pint units for Wadell formulas (closed-form takes plain ratios)
h = H_TO_GND.to(ureg.mm).magnitude
w = W_TRACE_USB.to(ureg.mm).magnitude
s = G_GAP_USB.to(ureg.mm).magnitude

z0_se = (60 / math.sqrt(EPSILON_R_FR4)) * math.log(8 * h / w + w / (4 * h)) * ureg.ohm
z_diff = 2 * z0_se * (1 - 0.48 * math.exp(-0.96 * s / h))

print(f"Stack-up: w={w} mm, gap={s} mm, h={h} mm to GND")
print(f"Z_0 (single-ended): {z0_se:.1f}")
print(f"Z_diff: {z_diff:.1f} (target {Z_DIFF_USB_TARGET} +/-10%)")
tol_low = Z_DIFF_USB_TARGET * 0.9
tol_high = Z_DIFF_USB_TARGET * 1.1
print(f"In tolerance ({tol_low:.0f}-{tol_high:.0f}): {tol_low < z_diff < tol_high}")

# Skew length budget
skew_length = (USB_SKEW_MAX * V_PROP_FR4).to(ureg.mm)
print(f"\nMax intra-pair skew length: {skew_length:.1f} ({USB_SKEW_MAX})")

Stack-up: w=0.2 mm, gap=0.15 mm, h=0.2 mm to GND
Z_0 (single-ended): 61.1 ohm
Z_diff: 93.6 ohm (target 90 ohm +/-10%)
In tolerance (81 ohm-99 ohm): True

Max intra-pair skew length: 14.5 millimeter (100 picosecond)


## Section 6: Anemometer Selection — Wind-Sensitivity-Driven Spec

The embedded engineer's pick (Calypso ULP STD) is not validated against the IEEE 738 sensitivity. This section derives the *required* sensor accuracy from the ampacity calculation, then matches candidates against it.

### Worst-case operating point (IEEE 738-2012 Annex A / `ieee738_test.py:114`)

- ACSR Drake 26/7, D = 28.14 mm, eps = 0.5, alpha = 0.5
- T_c = 100 °C (max), T_a = 40 °C, q_s = 0
- V_w = 0.61 m/s perpendicular (phi = 90°)

### Forced-convection scaling (low Re regime)

For Re < 4000 Hilpert gives Nu = 0.683 . Re^0.466 → q_c ∝ V^0.466. Differentiating:

$$ \frac{dq_c}{dV_w} = 0.466 \cdot \frac{q_c}{V_w} $$

Ampacity from heat balance $I^2 R = q_c + q_r - q_s$ gives:

$$ \frac{dI}{dV_w} = \frac{1}{2 I R(T_c)} \cdot \frac{dq_c}{dV_w} $$

This is the "expected value" the numerical sensitivity must match within order-of-magnitude.

### Wind-angle scaling (parallel-wind edge case)

$k_{angle}(\phi) = 1.194 - \cos\phi + 0.194\cos 2\phi + 0.368\sin 2\phi$, so:

$$ \frac{dk_{angle}}{d\phi} = \sin\phi - 0.388\sin 2\phi + 0.736\cos 2\phi $$

Sensitivity is small at perpendicular (φ=90°, dk/dφ ≈ 0.264 rad⁻¹) and large near parallel (φ=10°, dk/dφ ≈ 0.733 rad⁻¹) — direction accuracy matters most when wind is nearly parallel to the conductor.

In [7]:
# Hand calc: ampacity sensitivity to wind speed at IEEE 738 worst-case op point.
# "See the test fail" expected value before any numerical sim.
#
# Temperatures kept in Kelvin (multiplicative pint-safe). degC kept as plain floats
# only for the air-property polynomial fits — those fits are defined on Celsius scale.

# === IEEE 738 Annex A operating point ===
D_DRAKE: Final = 28.14 * ureg.mm
EPS_DRAKE: Final = 0.5
R_25C_DRAKE: Final = 7.283e-5 * ureg.ohm / ureg.m
ALPHA_T_DRAKE: Final = 0.00403 / ureg.kelvin  # per-degree, kelvin == delta_degC
T_C_PLAIN: Final = 100.0  # °C, plain float — used in poly fits
T_A_PLAIN: Final = 40.0  # °C
V_W_WORST: Final = 0.61 * ureg.m / ureg.s

T_C_K = (T_C_PLAIN + 273.15) * ureg.kelvin
T_A_K = (T_A_PLAIN + 273.15) * ureg.kelvin
T_FILM_PLAIN = 0.5 * (T_C_PLAIN + T_A_PLAIN)  # °C
T_FILM_K_PLAIN = T_FILM_PLAIN + 273.15
DELTA_T_K = (T_C_PLAIN - T_A_PLAIN) * ureg.kelvin

# Air properties at film temp (polynomial fits good to +/-1% per IEEE 738)
rho_air = (
    (1.293 - 1.525e-4 * T_FILM_PLAIN + 6.379e-9 * T_FILM_PLAIN**2) * ureg.kg / ureg.m**3
)
mu_air = (1.458e-6 * T_FILM_K_PLAIN**1.5) / (T_FILM_K_PLAIN + 110.4) * ureg.Pa * ureg.s
k_air = (
    (2.424e-2 + 7.477e-5 * T_FILM_PLAIN - 4.407e-9 * T_FILM_PLAIN**2)
    * ureg.W
    / ureg.m
    / ureg.K
)

# Reynolds number at worst-case V_w
re = (D_DRAKE * rho_air * V_W_WORST / mu_air).to(ureg.dimensionless).magnitude
print(
    f"Re = {re:.0f}  (regime: {'high (Nu=0.193 Re^0.618)' if re > 4000 else 'low (Nu=0.683 Re^0.466)'})"
)

# Nusselt (low-Re Hilpert) and forced convection per meter (phi = 90°, k_angle = 1.0)
nu = 0.683 * re**0.466 if re <= 4000 else 0.193 * re**0.618
q_forced = (math.pi * nu * k_air * DELTA_T_K).to(ureg.W / ureg.m)
print(f"Nu = {nu:.2f}")
print(f"q_forced (perpendicular) = {q_forced:.1f}")

# Natural convection — must check it doesn't dominate at this low V_w
q_natural = (
    (
        3.645
        * rho_air.magnitude**0.5
        * D_DRAKE.to(ureg.m).magnitude ** 0.75
        * (T_C_PLAIN - T_A_PLAIN) ** 1.25
    )
    * ureg.W
    / ureg.m
)
print(
    f"q_natural = {q_natural:.1f}  ({'natural dominates!' if q_natural > q_forced else 'forced dominates'})"
)
q_c = q_forced if q_forced > q_natural else q_natural

# Radiative loss
sigma = 5.6697e-8 * ureg.W / ureg.m**2 / ureg.K**4
q_r = (math.pi * D_DRAKE * EPS_DRAKE * sigma * (T_C_K**4 - T_A_K**4)).to(
    ureg.W / ureg.m
)
print(f"q_r = {q_r:.1f}")

# Resistance at T_c
r_tc = (R_25C_DRAKE * (1 + ALPHA_T_DRAKE * (T_C_PLAIN - 25.0) * ureg.kelvin)).to(
    ureg.ohm / ureg.m
)
print(f"R(T_c) = {r_tc.to(ureg.uohm / ureg.m):.1f}")

# Ampacity (q_s = 0 per Annex A worst-case)
net_cooling = q_c + q_r
i_max_hand = ((net_cooling / r_tc) ** 0.5).to(ureg.A)
print(f"\nI_max (hand calc) = {i_max_hand:.0f}")

# Sensitivity: q_c proportional to V^0.466 in low-Re regime → dq_c/dV = 0.466 . q_c / V
dqc_dv = (0.466 * q_c / V_W_WORST).to(ureg.W / ureg.m / (ureg.m / ureg.s))
print(f"dq_c/dV_w (analytical) = {dqc_dv:.1f}")

# dI/dV = (1/(2.I.R)) . dq_c/dV
di_dv_hand = (dqc_dv / (2 * i_max_hand * r_tc)).to(ureg.A / (ureg.m / ureg.s))
print(f"dI/dV_w (hand calc) = {di_dv_hand:.1f}")

# Required speed accuracy for 5% ampacity tolerance from wind term alone
TARGET_AMPACITY_TOL: Final = 0.05  # 5% of I_max — utility DLR practice
delta_i_budget = TARGET_AMPACITY_TOL * i_max_hand
delta_v_required = (delta_i_budget / di_dv_hand).to(ureg.m / ureg.s)
print("\n=== Derived speed-accuracy spec ===")
print(
    f"Ampacity tolerance budget: +/- {delta_i_budget:.0f} ({TARGET_AMPACITY_TOL * 100:.0f}% of I_max)"
)
print(
    f"Required wind-speed accuracy at V=0.61 m/s: +/- {delta_v_required.to(ureg.cm / ureg.s):.1f}"
    f"  (= +/- {delta_v_required.magnitude:.3f} m/s)"
)

Re = 1077  (regime: low (Nu=0.683 Re^0.466))
Nu = 17.68
q_forced (perpendicular) = 98.2 watt / meter
q_natural = 47.4 watt / meter  (forced dominates)
q_r = 24.5 watt / meter
R(T_c) = 94.8 microohm / meter

I_max (hand calc) = 1137 ampere
dq_c/dV_w (analytical) = 75.0 second * watt / meter ** 2
dI/dV_w (hand calc) = 347.6 ampere * second / meter

=== Derived speed-accuracy spec ===
Ampacity tolerance budget: +/- 57 ampere (5% of I_max)
Required wind-speed accuracy at V=0.61 m/s: +/- 16.4 centimeter / second  (= +/- 0.164 m/s)


In [8]:
# Numerical sensitivity using the *actual* IEEE 738 implementation that
# dlr-operating-envelope will run. Finite-difference dI/dV_w and dI/dphi at
# multiple operating points. Hand-calc above is the "expected value" — must
# match within ~10% (centered diff of smooth function).
#
# Sibling repo layout: ../dlr-operating-envelope/src/ieee738.py
import sys
from pathlib import Path

DLR_ENV_PATH = Path("../dlr-operating-envelope").resolve()
if str(DLR_ENV_PATH) not in sys.path:
    sys.path.insert(0, str(DLR_ENV_PATH))

from src.ieee738 import DRAKE_ACSR_795, steady_state_current  # noqa: E402

# --- Confirm hand-calc op point against production code ---
i_max_prod = steady_state_current(
    conductor=DRAKE_ACSR_795,
    conductor_temp_c=100.0,
    ambient_temp_c=40.0,
    wind_speed_mps=0.61,
    solar_irradiance_w_per_m2=0.0,
    wind_angle_deg=90.0,
)
print(f"I_max (production code) = {i_max_prod:.0f} A")
print(f"I_max (hand calc)       = {i_max_hand.magnitude:.0f} A")
agreement = abs(i_max_prod - i_max_hand.magnitude) / i_max_prod
print(f"Agreement: {(1 - agreement) * 100:.1f}%  (must be > 90%)")
assert agreement < 0.10, (
    f"Hand calc and production disagree by {agreement * 100:.1f}% — investigate before trusting derived spec"
)


# --- Finite-difference dI/dV_w at the worst-case op point ---
def di_dv(v_mps: float, phi_deg: float = 90.0, dv: float = 0.01) -> float:
    """Centered-difference dI/dV_w (A per m/s)."""
    i_plus = steady_state_current(DRAKE_ACSR_795, 100.0, 40.0, v_mps + dv, 0.0, phi_deg)
    i_minus = steady_state_current(
        DRAKE_ACSR_795, 100.0, 40.0, max(v_mps - dv, 0.001), 0.0, phi_deg
    )
    return (i_plus - i_minus) / (2 * dv)


def di_dphi(v_mps: float, phi_deg: float, dphi: float = 0.5) -> float:
    """Centered-difference dI/dphi (A per deg)."""
    i_plus = steady_state_current(
        DRAKE_ACSR_795, 100.0, 40.0, v_mps, 0.0, phi_deg + dphi
    )
    i_minus = steady_state_current(
        DRAKE_ACSR_795, 100.0, 40.0, v_mps, 0.0, phi_deg - dphi
    )
    return (i_plus - i_minus) / (2 * dphi)


di_dv_num = di_dv(0.61, 90.0)
print("\n--- dI/dV_w at worst-case (0.61 m/s, perpendicular) ---")
print(f"Numerical: {di_dv_num:.1f} A/(m/s)")
print(f"Hand calc: {di_dv_hand.magnitude:.1f} A/(m/s)")
match_v = abs(di_dv_num - di_dv_hand.magnitude) / di_dv_num
print(f"Match: {(1 - match_v) * 100:.1f}%")

# --- Sensitivity scan: dI/dV across operating envelope ---
print("\n--- dI/dV_w across operating envelope (phi=90°) ---")
print(f"{'V (m/s)':>10} {'I (A)':>8} {'dI/dV (A/(m/s))':>18}")
for v in [0.5, 1.0, 2.0, 5.0, 10.0]:
    i_at_v = steady_state_current(DRAKE_ACSR_795, 100.0, 40.0, v, 0.0, 90.0)
    print(f"{v:>10.2f} {i_at_v:>8.0f} {di_dv(v):>18.1f}")

# --- Direction sensitivity scan: worst when wind nearly parallel ---
print("\n--- dI/dphi across angle (V=0.61 m/s) ---")
print(f"{'phi (deg)':>10} {'I (A)':>8} {'dI/dphi (A/deg)':>18}")
for phi in [10.0, 30.0, 45.0, 60.0, 90.0]:
    i_at_phi = steady_state_current(DRAKE_ACSR_795, 100.0, 40.0, 0.61, 0.0, phi)
    print(f"{phi:>10.1f} {i_at_phi:>8.0f} {di_dphi(0.61, phi):>18.2f}")

I_max (production code) = 1137 A
I_max (hand calc)       = 1137 A
Agreement: 100.0%  (must be > 90%)

--- dI/dV_w at worst-case (0.61 m/s, perpendicular) ---
Numerical: 347.6 A/(m/s)
Hand calc: 347.6 A/(m/s)
Match: 100.0%

--- dI/dV_w across operating envelope (phi=90°) ---
   V (m/s)    I (A)    dI/dV (A/(m/s))
      0.50     1096              401.1
      1.00     1249              243.0
      2.00     1435              146.2
      5.00     1833              104.6
     10.00     2240               65.7

--- dI/dphi across angle (V=0.61 m/s) ---
 phi (deg)    I (A)    dI/dphi (A/deg)
      10.0      891               7.43
      30.0     1014               4.74
      45.0     1069               2.70
      60.0     1098               1.33
      90.0     1137               2.10


In [9]:
# Derived sensor spec + power-budget impact.
# Pulls direction-accuracy from the parallel-wind worst case found in the sensitivity scan.

# --- Direction-accuracy spec: worst-case is at phi=10° (nearly parallel) ---
PHI_WORST: Final = 10.0  # deg, where dk_angle/dphi is largest
i_at_phi_worst = steady_state_current(DRAKE_ACSR_795, 100.0, 40.0, 0.61, 0.0, PHI_WORST)
di_dphi_worst = di_dphi(0.61, PHI_WORST)
delta_i_budget_phi = TARGET_AMPACITY_TOL * i_at_phi_worst * ureg.A
delta_phi_required = delta_i_budget_phi.magnitude / abs(di_dphi_worst)  # degrees
print(
    f"At phi={PHI_WORST}°: I={i_at_phi_worst:.0f} A, dI/dphi={di_dphi_worst:.2f} A/deg"
)
print(
    f"Required direction accuracy: +/- {delta_phi_required:.1f} deg "
    f"(for 5% of I={i_at_phi_worst:.0f} A)"
)

# --- Power-budget impact: continuous 100 mW ultrasonic sensor ---
P_SENSOR_CONT: Final = 100 * ureg.mW  # FT205 / Calypso / Gill class
e_sensor_daily = (P_SENSOR_CONT * 24 * ureg.hour).to(ureg.W * ureg.hour)
e_daily_with_sensor = e_daily + e_sensor_daily
winter_margin_new = (e_pv_winter / e_daily_with_sensor).to(ureg.dimensionless)
days_autonomy_new = (e_bat_usable / e_daily_with_sensor).to(ureg.dimensionless)

print("\n=== Power-budget delta: unheated ultrasonic (100 mW continuous) ===")
print(f"Sensor daily energy:           {e_sensor_daily:.2f}")
print(f"Daily energy (existing):       {e_daily.to(ureg.W * ureg.hour):.1f}")
print(
    f"Daily energy (with sensor):    {e_daily_with_sensor.to(ureg.W * ureg.hour):.1f}"
)
print(f"PV winter:                     {e_pv_winter.to(ureg.W * ureg.hour):.1f}")
print(f"Winter margin (was 1.55x):     {winter_margin_new.magnitude:.2f}x")
print(f"Autonomy (was 1.59 days):      {days_autonomy_new.magnitude:.2f} days")

# --- Power-budget impact: heated ultrasonic (e.g. Lufft V200A, 25W during ice) ---
P_HEATER: Final = 25 * ureg.W
T_HEATER_PER_DAY: Final = 2 * ureg.hour  # worst-case winter assumption
e_heater_daily = (P_HEATER * T_HEATER_PER_DAY).to(ureg.W * ureg.hour)
e_daily_heated = e_daily + e_sensor_daily + e_heater_daily
winter_margin_heated = (e_pv_winter / e_daily_heated).to(ureg.dimensionless)

print("\n=== Power-budget delta: heated ultrasonic (25W heater, 2h/day winter) ===")
print(f"Heater daily energy:           {e_heater_daily:.1f}")
print(f"Daily energy (heated case):    {e_daily_heated.to(ureg.W * ureg.hour):.1f}")
print(
    f"Winter margin:                 {winter_margin_heated.magnitude:.2f}x"
    f"  ({'VIABLE' if winter_margin_heated.magnitude > 1.0 else 'NOT VIABLE on current PV+battery'})"
)

At phi=10.0°: I=891 A, dI/dphi=7.43 A/deg
Required direction accuracy: +/- 6.0 deg (for 5% of I=891 A)

=== Power-budget delta: unheated ultrasonic (100 mW continuous) ===
Sensor daily energy:           2.40 hour * watt
Daily energy (existing):       29.1 hour * watt
Daily energy (with sensor):    31.5 hour * watt
PV winter:                     45.0 hour * watt
Winter margin (was 1.55x):     1.43x
Autonomy (was 1.59 days):      1.46 days

=== Power-budget delta: heated ultrasonic (25W heater, 2h/day winter) ===
Heater daily energy:           50.0 hour * watt
Daily energy (heated case):    81.5 hour * watt
Winter margin:                 0.55x  (NOT VIABLE on current PV+battery)


## Section 6 — Candidate Comparison vs Derived Spec

### Derived spec from above

| Param | Spec | Source |
|---|---|---|
| Speed accuracy (V < 5 m/s) | **±0.16 m/s** | dI/dV = 348 A/(m/s) at V=0.61 m/s, 5 % I_max budget |
| Speed threshold | **≤ 0.5 m/s** | Must measure IEEE 738 Annex A op point (V=0.61 m/s) |
| Direction accuracy | **±6°** | dI/dφ = 7.4 A/deg at φ=10°, 5 % I_max budget |
| Speed range | 0.5 – 30 m/s | Storm range not needed; DLR upside is in low wind |
| Power (avg) | < 0.5 W | 100 mW = 2.4 Wh/day = 5 % of winter PV harvest |
| Operating temp (sensor) | -20 to +50 °C | Sensor in clean airflow on cross-arm; not enclosure +85 °C |
| Interface | Differential serial (RS-485 / SDI-12 / Modbus RTU) | I2C/single-ended fragile over mast cable in 60 Hz B-field |
| Heating | If selected, on-demand only (icing self-test) | Continuous heating breaks power budget — see calc above |

### Candidate matrix (sourced datasheet figures)

| Sensor | Threshold | Speed acc | Direction acc | Op temp | Interface | Power | Verdict |
|---|---|---|---|---|---|---|---|
| **Calypso ULP STD** (embedded engineer's pick) | **1.0 m/s** | ±0.1 m/s **@ 10 m/s** (low-V unspec) | ±1° | unspecified | RS-485/UART/Modbus/NMEA | **1.25 mW** | **FAIL threshold** — cannot read 0.61 m/s |
| **FT Technologies FT205** | not spec'd | **±0.3 m/s** (0-16 m/s) | 4° RMS | **-20 to +70 °C** | RS-485/UART | up to 200 mA | **FAIL speed accuracy** at low V |
| **Gill WindObserver 65** | 0.01 m/s | ±2 % (≈0.012 m/s @ 0.61) | ±2° | -55 to +70 °C (heated opt to -55) | RS-422/485/SDI-12/Modbus/NMEA | 360 mW (40 mA @ 9-30 V) + opt 5 W or 72 W heater | **PASS** |
| **Vaisala WMT702** | 0.01 m/s | ±0.1 m/s OR 2 % whichever greater | ±2° (typ) | -52 to +60 °C | RS-232/422/485/SDI-12 | ~500 mW (no heat) | **PASS** |

Calypso threshold consequence: at true V_w = 0.61 m/s the sensor reports 0, the system falls back to natural-convection-only cooling, q_c drops from 98.2 → 47.4 W/m, ampacity drops from 1137 → 871 A — a **23 % systematic under-estimate** in the exact regime where DLR is most valuable. Safe direction, but defeats the project's purpose.

### Two paths forward — real decision gate

**Path A — Strict spec, full DLR uplift.** Vaisala WMT702 ($2 800) or Gill WindObserver 65 ($1 500). Meets every derived spec. Cost: 5–10× the embedded engineer's pick; power 4–10× higher (still inside 1.5× winter margin once the 50 Wh battery is sized for 1 day autonomy instead of 1.6 days). Captures DLR headroom across the full IEEE 738 operating envelope.

**Path B — Pragmatic spec, DLR suspended at low V.** Keep a Calypso-class ultra-low-power sensor and re-define the system: below V_w = 1.0 m/s the operating envelope falls back to the conductor's static thermal rating. This matches how production utility DLR deployments handle the low-wind regime (the marginal headroom at <1 m/s is small anyway, and direction is unreliable at near-zero wind). Cost: ~$300; power 1 mW (~0.001 % of budget). Loses the bottom slice of the IEEE 738 capacity curve.

**Tradeoff** — Path A captures ~10–20 % more ampacity headroom in the V_w ∈ [0.1, 1.0] m/s regime; Path B is 5–10× cheaper, 100× lower power, and matches industry practice. The right answer depends on whether the project's commercial value model relies on capturing that bottom slice. **Need PM input.**

### SKU split — per-site climatology threshold

Mass-deployed: pick variant from site wind climatology, not one-size-fits-all.

Decision metric: $P_{\text{low-wind}} = \Pr(V_{\perp} < 1 \text{ m/s})$ over a representative year.

- $V_{\perp} = V_w \cdot |\sin(\phi_{wd} - \phi_{cond})|$
- $V_w$ Weibull-distributed: $f(V) = \frac{k}{c}\left(\frac{V}{c}\right)^{k-1} e^{-(V/c)^k}$  (k=2 ≈ Rayleigh for free-stream winds)
- $\phi_{wd}$ assumed uniform over $[0, 2\pi]$ — worst case; real sites have prevailing wind that tightens the variance but does not shift the mean fraction much
- Threshold: $P_{\text{low-wind}} > 0.15$ → **low-wind variant** (Vaisala/Gill); else **high-wind variant** (Calypso)

Monte Carlo over 1 yr of hourly samples (N=8760) per site class.

In [10]:
# SKU-split methodology demo: Weibull wind + uniform direction -> P(V_perp < 1 m/s).
# Uses synthetic site climates; real deployment pulls per-tower data from
# NREL WIND Toolkit / ERA5 / utility SCADA.
import numpy as np

V_PERP_THRESHOLD: Final = 1.0 * ureg.m / ureg.s  # Calypso threshold
SKU_THRESHOLD: Final = 0.15  # >15 % yr -> low-wind variant
N_HOURS: Final = 8760  # 1 yr hourly samples
WEIBULL_K: Final = 2.0  # Rayleigh-like; standard for free-stream

# Three synthetic site classes — Weibull scale parameter c (m/s).
# Mean wind = c * gamma(1 + 1/k); for k=2, gamma(1.5) ~ 0.886, so mean ~ 0.886 * c
SITE_CLASSES = {
    "valley / sheltered": 3.0,
    "open plain": 5.0,
    "ridge / coastal": 8.0,
}

rng = np.random.default_rng(seed=42)

print(
    f"{'Site class':>22}  {'mean V (m/s)':>14}  {'P(V_perp<1 m/s)':>18}  {'Variant':>22}"
)
print("-" * 80)
for label, c in SITE_CLASSES.items():
    # Weibull sample for wind speed magnitude
    v_w = c * rng.weibull(WEIBULL_K, N_HOURS)
    # Uniform wind direction relative to conductor axis
    phi = rng.uniform(0, 2 * np.pi, N_HOURS)
    v_perp = v_w * np.abs(np.sin(phi))
    p_lowwind = float((v_perp < V_PERP_THRESHOLD.magnitude).mean())
    variant = (
        "low-wind (Vaisala/Gill)"
        if p_lowwind > SKU_THRESHOLD
        else "high-wind (Calypso)"
    )
    print(f"{label:>22}  {v_w.mean():>14.2f}  {p_lowwind:>18.2%}  {variant:>22}")

            Site class    mean V (m/s)     P(V_perp<1 m/s)                 Variant
--------------------------------------------------------------------------------
    valley / sheltered            2.64              35.72%  low-wind (Vaisala/Gill)
            open plain            4.42              22.60%  low-wind (Vaisala/Gill)
       ridge / coastal            7.13              13.61%     high-wind (Calypso)


## Expected Values

| Parameter | Value | Tolerance | Source |
|-----------|-------|-----------|--------|
| Buck input current at peak (V_BAT_MIN) | 2.18 A | +/-0.07 A | Power conservation, η=0.90+/-0.03 |
| Buck input current typical (V_BAT_NOM) | 0.72 A | +/-0.02 A | Power conservation |
| Battery 1C limit (4 Ah cell) | 4.0 A | — | Datasheet |
| LDO V_in headroom at 3V3 | 1.45 V | — | AP2112K dropout 250 mV at 600 mA |
| Daily average current (5V) | 242 mA | — | Realistic duty cycle |
| Daily energy budget (no anemometer) | 29.1 Wh | — | I_avg × 5V × 24h |
| Daily energy budget (with 100 mW anemometer) | 31.5 Wh | — | +2.4 Wh for continuous sensor draw |
| PV production (winter worst) | 45.0 Wh/day | — | 20W × 2.5 h × 0.90 MPPT η |
| Winter margin (no anemometer) | 1.55x | — | E_pv / E_daily |
| Winter margin (100 mW anemometer) | 1.43x | — | Sensor consumes ~8 % of margin |
| Winter margin (25 W heater 2 h/day) | 0.55x | — | **NOT VIABLE** on current 20W panel |
| Battery autonomy (50 Wh, 90% DoD, no sensor) | 1.59 days | — | E_bat_usable / E_daily |
| Battery autonomy (with 100 mW sensor) | 1.46 days | — | Acceptable; rec resize to 100 Wh per Section 2 |
| I2C pull-up | 2.2 kΩ | E24 | UM10204 Table 10, in 967Ω–4.72kΩ range |
| I2C rise time at selected R | 140 ns | — | < 300 ns spec, 53% margin |
| SPI critical trace length | 254 mm | — | TL threshold; design rule keep < 51 mm |
| USB Z_diff | 93.6 Ω | target 90 +/-9 Ω | Wadell, 8/6 mil on h=0.2mm |
| USB skew length budget | 14.5 mm | — | 100 ps × v_prop |
| **Ampacity at IEEE 738 worst-case op pt** (V=0.61 m/s, φ=90°, T_c=100°C, T_a=40°C, q_s=0) | **1137 A** | — | Hand calc = production code, 100% agreement |
| **dI/dV_w at worst-case op point** | **348 A/(m/s)** | — | Hand calc and numerical centered-diff agree exactly |
| **dI/dφ at parallel wind (φ=10°, V=0.61)** | **7.4 A/deg** | — | Worst-case direction sensitivity |
| **Derived sensor speed accuracy spec** | **±0.16 m/s at V<1 m/s** | — | 5 % I_max budget / dI/dV at worst-case |
| **Derived sensor speed threshold spec** | **≤ 0.5 m/s** | — | Must measure IEEE 738 Annex A worst-case (0.61 m/s) |
| **Derived sensor direction accuracy spec** | **±6°** | — | 5 % I_max budget / dI/dφ at φ=10° |

**Design notes:**

- **Battery sizing:** 50 Wh autonomy is 1.59 days at zero PV — winter cloud weeks could deplete. Recommend stepping to **100 Wh battery** for ~3 days autonomy.
- **Buck topology:** must accept V_in = 10.0 V to 14.6 V continuously (1.46x range). Synchronous buck with low-Rds_on FETs preferred for the 0.90 efficiency assumption.
- **USB diff impedance** assumes a controlled-impedance fab order with the specific 4-layer stack (h=0.20mm to GND). Generic 1.6mm fab will not hit tolerance.
- **I2C bus capacitance** estimate (75 pF) covers 4 device slots — verify after layout, re-derive if exceeded.

**Anemometer decision gate — sensor selection is BLOCKED on PM input:**

- The embedded engineer's Calypso ULP STD pick **fails the derived spec** — its 1.0 m/s threshold cannot measure the IEEE 738 worst-case op point.
- Two viable paths surface from the analysis: (A) strict-spec sensor (Vaisala WMT702 / Gill WindObserver 65, $1.5–3 k, ~500 mW) captures full DLR uplift; (B) low-power Calypso-class sensor (~$300, 1 mW) requires the system to formally suspend DLR below V_w = 1 m/s and fall back to static rating.
- See Section 6 candidate matrix and tradeoff discussion. **No hardware integration or sim work should proceed until path A vs B is chosen.**